In [43]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix

In [44]:
# TODO: load the glass dataset into df
df = pd.read_csv("glass.csv")

# view first few rows
df.head()

,RI,Na,Mg,Al,Si,K,Ca,Ba,Fe,Type
0,1.52101,13.64,4.49,1.10,71.78,0.06,8.75,0.0,0.0,1
1,1.51761,13.89,3.60,1.36,72.73,0.48,7.83,0.0,0.0,1
2,1.51618,13.53,3.55,1.54,72.99,0.39,7.78,0.0,0.0,1
3,1.51766,13.21,3.69,1.29,72.61,0.57,8.22,0.0,0.0,1
4,1.51742,13.27,3.62,1.24,73.08,0.55,8.07,0.0,0.0,1


In [45]:
# TODO: check number of rows and columns
df.shape


(214, 10)

In [46]:
# TODO: see column names
df.columns


Index(['RI', 'Na', 'Mg', 'Al', 'Si', 'K', 'Ca', 'Ba', 'Fe', 'Type'], dtype='object')

In [47]:
# TODO: look at first few rows
df.head()


,RI,Na,Mg,Al,Si,K,Ca,Ba,Fe,Type
0,1.52101,13.64,4.49,1.10,71.78,0.06,8.75,0.0,0.0,1
1,1.51761,13.89,3.60,1.36,72.73,0.48,7.83,0.0,0.0,1
2,1.51618,13.53,3.55,1.54,72.99,0.39,7.78,0.0,0.0,1
3,1.51766,13.21,3.69,1.29,72.61,0.57,8.22,0.0,0.0,1
4,1.51742,13.27,3.62,1.24,73.08,0.55,8.07,0.0,0.0,1


Displays dataset size, column names, and sample rows.

Why it is needed:
Understanding the structure of the dataset ensures correct identification of input features and output labels.

In [48]:
df["y"] = (df["Type"] == 1).astype(int)
df = df.drop(columns=["Type"])

Converts the multi-class glass type into a binary output variable where Type-1 glass is labeled as 1 and all others as 0.

Why needed: Logistic regression is a binary classification model and requires binary labels.


In [49]:
# TODO: separate features and labels
X = df.drop(columns=["y"]).values
y = df["y"].values


Separates input features (X) and output labels (y).

Why it is needed:
The model must know what to learn from and what to predict.

In [50]:
# TODO: split data into train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


Splits data into training and testing sets.

Why it is needed:
Testing on unseen data checks generalization, not memorization.

In [51]:
# TODO: scale features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


Normalizes feature values to a similar scale.

Why it is needed:
Sigmoid is sensitive to large values; scaling ensures stable learning

In [52]:
# TODO: return sigmoid of z
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


Converts a linear score into a probability between 0 and 1.

Why it is needed:
Sigmoid enables soft, probabilistic decisions instead of hard labels.

In [53]:
# TODO: compute probability output
def predict_proba(X, w, b):
    z = X @ w + b          # compute linear score
    p = sigmoid(z)         # convert to probability
    return p


Computes the probability of belonging to the positive class.

Why it is needed:
Probabilities preserve confidence and distance from the decision boundary.

In [54]:
# TODO: compute binary cross entropy loss
def loss(y, p):
    return -np.mean(y * np.log(p + 1e-9) + (1 - y) * np.log(1 - p + 1e-9))


Calculates binary cross-entropy loss.

Why it is needed:
Penalizes confident wrong predictions more and guides learning.

In [55]:
# TODO: update weights and bias
def update_weights(X, y, w, b, lr):
    p = predict_proba(X, w, b)     # predictions
    error = p - y                 # error

    w = w - lr * (X.T @ error) / len(y)
    b = b - lr * np.mean(error)

    return w, b


Updates weights and bias using gradient descent.

Why it is needed:
Adjusting parameters reduces prediction error.

In [56]:
# TODO: initialize weights and bias
w = np.zeros(X_train.shape[1])
b = 0.0

lr = 0.1
epochs = 100

# training loop
for _ in range(epochs):
    w, b = update_weights(X_train, y_train, w, b, lr)


Trains the model over multiple epochs.

Why it is needed:
Repeated updates help the model converge to better parameters.

In [57]:
# TODO: convert probability to label
def predict_label(p, threshold=0.5):
    return (p >= threshold).astype(int)


Converts probabilities into class labels using a threshold.

Why it is needed:
Thresholds control how conservative or aggressive decisions are.

In [58]:
# predictions
p_test = predict_proba(X_test, w, b)
y_pred = predict_label(p_test, threshold=0.5)

print("Accuracy (threshold=0.5):", accuracy_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


Accuracy (threshold=0.5): 0.8604651162790697
Confusion Matrix:
 [[30  2]
 [ 4  7]]


Evaluates accuracy and confusion matrix.

Why it is needed:
These metrics show how well the model performs.

In [59]:
for t in [0.5, 0.7]:
    y_pred_t = predict_label(p_test, threshold=t)
    print(f"\nThreshold = {t}")
    print("Accuracy:", accuracy_score(y_test, y_pred_t))



Threshold = 0.5
Accuracy: 0.8604651162790697

Threshold = 0.7
Accuracy: 0.7209302325581395


Evaluates performance at different probability thresholds.

Why it is needed:
Different applications require different trade-offs between errors.

1. How this differs from perceptron
2. Why sigmoid matters
3. What problem still remains unsolved


Logistic regression differs from the perceptron because it outputs probabilities instead of hard binary decisions. A perceptron uses a step function that forces outputs to be either 0 or 1, while logistic regression applies a sigmoid function to express confidence in predictions. The sigmoid function is important because it preserves information about how close a data point is to the decision boundary and provides stable learning under noisy conditions. However, logistic regression still uses a linear decision boundary, so it cannot solve non-linearly separable problems such as XOR.